# LIMIT — Multi-Model Evaluation

In [ ]:
from evaluate import eval_item_retrieval
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# query_prefix: prepended to every query before encoding (check model card if unsure)
# Local (fits in 8 GB VRAM)
MODELS = {
    "GTE-ModernBERT": {
        "hf_id": "Alibaba-NLP/gte-modernbert-base",
        "query_prefix": "query: ",
    },
    "Snowflake-Arctic-L": {
        "hf_id": "Snowflake/snowflake-arctic-embed-l",
        "query_prefix": "",
    },
    "Qwen3-Embedding": {
        "hf_id": "Qwen/Qwen3-Embedding",
        "query_prefix": "query: ",
    },
    # Large — run on Colab (need >8 GB VRAM)
    # "E5-Mistral-7B": {
    #     "hf_id": "intfloat/e5-mistral-7b-instruct",
    #     "query_prefix": "Instruct: Retrieve the person whose profile contains the queried item.\nQuery: ",
    # },
    # "GritLM-7B": {
    #     "hf_id": "GritLM/GritLM-7B",
    #     "query_prefix": "",
    # },
    # "Promptriever-Llama3-8B": {
    #     "hf_id": "samaya-ai/promptriever-llama3.1-8b-instruct-v1",
    #     "query_prefix": "",
    # },
}

In [ ]:
MODEL_NAME = "GTE-ModernBERT"  # change to any key in MODELS

In [ ]:
N = 100
if "all_results" not in globals():
    all_results = {}

cfg = MODELS[MODEL_NAME]
print(f"\n{'='*60}\n{MODEL_NAME}")
all_results[MODEL_NAME] = eval_item_retrieval(
    n=N,
    model_name=cfg["hf_id"],
    query_prefix=cfg["query_prefix"],
)

In [ ]:
def plot_model_comparison(all_results, metric="recall@1"):
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = cm.tab10.colors
    for i, (label, results) in enumerate(all_results.items()):
        ms = sorted(results)
        scores = [results[m][metric] for m in ms]
        ax.plot(ms, scores, marker="o", label=label, color=colors[i % len(colors)])
    ax.set_xlabel("m (items per person)")
    ax.set_ylabel(metric)
    ax.set_title(f"Model comparison — {metric}")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_model_comparison(all_results, "recall@1")
plot_model_comparison(all_results, "mrr")